# TorGen: DETR-CVAE for Tornado Outbreak Generation

Install the package, mount Drive, and train.

In [ ]:
!pip install -q git+https://github.com/jcaramichaellehigh/TorGen.git
!pip show torgen | grep -E "^(Version|Location)"
!python -c "
from torgen.loss.hungarian import _focal_bce
from torgen.data.dataset import coords_to_bearing_length
from torgen.model.decoder import TrackDecoder
import inspect
src = inspect.getsource(TrackDecoder.forward)
assert 'z_broadcast' in src, 'MISSING: z injection'
assert 'memory_drop' in src, 'MISSING: memory dropout'
from torgen.training.config import TrainConfig
cfg = TrainConfig(drive_dir='', checkpoint_dir='')
assert cfg.lambda_noobj == 2.0, f'lambda_noobj={cfg.lambda_noobj}, expected 2.0'
assert cfg.n_decoder_layers == 2, f'n_decoder_layers={cfg.n_decoder_layers}, expected 2'
assert cfg.memory_dropout == 0.2, f'memory_dropout={cfg.memory_dropout}, expected 0.2'
print('All fixes confirmed: focal loss, z-injection, bearing/length, memory dropout, lambda_noobj=2.0, decoder_layers=2')
"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Optional: experiment tracking
# Sign up at wandb.ai, get API key from wandb.ai/authorize
try:
    import wandb
    wandb.login()
except ImportError:
    print("wandb not installed, skipping experiment tracking")

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)

from torgen.training import TrainConfig, train

config = TrainConfig(
    drive_dir="/content/drive/MyDrive/raw",
    checkpoint_dir="/content/drive/MyDrive/checkpoints",
)

trainer = train(config)

In [ ]:
print(f"Epochs trained: {trainer.epoch}")
print(f"Final train loss: {trainer.train_losses[-1]:.4f}")
print(f"Final val loss: {trainer.val_losses[-1]:.4f}")
print(f"Best val loss: {trainer.best_val_loss:.4f}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(trainer.train_losses) + 1), trainer.train_losses, label="Train")
ax.plot(range(1, len(trainer.val_losses) + 1), trainer.val_losses, label="Val")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("TorGen Training Curves")
ax.legend()
fig.tight_layout()
plt.show()